# Phase 2 — primitive discovery

Fit the same whole-trial split to the observed trajectory `q` and endpoint residual `f`. FADA uses the local Fourier anechoic-demixing implementation; SCA calls the official Python package. This notebook does not select a winning representation or primitive model automatically.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_primitives.paths import PROJECT_ROOT
from motion_primitives.preprocessing import load_collections
from motion_primitives.phase2 import make_split, fit_fada, fit_sca

# ---- experiment settings ----
COLLECTION = PROJECT_ROOT / 'collections/3D-ARM-Gaze/custom-phase200-v2'
VIEW = 'successful'          # use 'all' to include failed trials
JOINT_NAMES = None
SEED = 42
COMPONENTS = [2, 3, 4, 5, 6]

RUN_FADA = True
FADA_MODEL = 'spatiotemporal'   # temporal | spatiotemporal | space_by_time
FADA_BACKEND = 'torch'
FADA_DEVICE = 'auto'
FADA_BATCH_SIZE = 512
FADA_MAX_DELAY = 20
FADA_RESTARTS = 2

RUN_SCA = True
SCA_EPOCHS = 3000

OUTPUT = PROJECT_ROOT / 'notebooks/results/03_Primitive_Discovery' / COLLECTION.name / VIEW
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
collection = load_collections([COLLECTION], JOINT_NAMES, view=VIEW)
Q, F = collection['q'], collection['f']
metadata = collection['metadata'].reset_index(drop=True)
joints = collection['joint_names']

split_path = OUTPUT / 'split.csv'
if split_path.exists():
    saved = pd.read_csv(split_path)
    if saved.trial_id.tolist() != metadata.trial_id.tolist():
        raise ValueError('Saved split does not match this ordered trial cohort.')
    split = saved.split.to_numpy()
else:
    split = make_split(metadata, seed=SEED)
    pd.DataFrame({'trial_id': metadata.trial_id, 'split': split}).to_csv(split_path, index=False)

display(pd.Series(split).value_counts().rename('trials').to_frame())
print(f'{len(metadata)} trials, {Q.shape[1]} phase samples, {Q.shape[2]} joints')


In [ ]:
provenance = {
    'collection': str(COLLECTION), 'view': VIEW, 'seed': SEED,
    'components': COMPONENTS, 'joint_names': joints,
    'representations': ['q', 'f'],
    'fada': {'model': FADA_MODEL, 'backend': FADA_BACKEND, 'device': FADA_DEVICE,
             'batch_size': FADA_BATCH_SIZE, 'max_delay': FADA_MAX_DELAY,
             'restarts': FADA_RESTARTS},
    'sca': {'epochs': SCA_EPOCHS},
}
(OUTPUT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')


## FADA

Each fit learns the primitive library only from `discovery` trials. Validation and test trials are projected onto that fixed library. The identical split is used for `q` and `f`.


In [ ]:
fada_rows = []
if RUN_FADA:
    for representation, values in [('q', Q), ('f', F)]:
        for k in COMPONENTS:
            print(f'FADA {representation=} {k=}')
            result = fit_fada(
                values, split, model=FADA_MODEL, n_components=k,
                backend=FADA_BACKEND, device=FADA_DEVICE,
                batch_size=FADA_BATCH_SIZE, max_delay=FADA_MAX_DELAY,
                restarts=FADA_RESTARTS, seed=SEED,
            )
            folder = OUTPUT / 'fada' / representation / f'k{k:02d}'
            folder.mkdir(parents=True, exist_ok=True)
            result['metrics'].to_csv(folder / 'metrics.csv', index=False)
            arrays = {'coefficients': result['coefficients'], 'delays': result['delays'],
                      'mean': result['mean'], 'history': result['history']}
            arrays.update({f'library_{name}': value for name, value in result['library'].items()})
            np.savez_compressed(folder / 'fit.npz', **arrays)
            for row in result['metrics'].to_dict('records'):
                fada_rows.append({'representation': representation, 'k': k,
                                  'model': FADA_MODEL, 'device': result['device'], **row})

fada_summary = pd.DataFrame(fada_rows)
if len(fada_summary):
    fada_summary.to_csv(OUTPUT / 'fada_summary.csv', index=False)
    display(fada_summary)


## Sparse Component Analysis

SCA is fit only on discovery samples after reshaping `(trial, phase, joint)` to `(sample, joint)`. Validation/test trajectories are transformed and reconstructed with the fitted model; no held-out sample is used for fitting.


In [ ]:
sca_rows = []
if RUN_SCA:
    for representation, values in [('q', Q), ('f', F)]:
        for k in COMPONENTS:
            print(f'SCA {representation=} {k=}')
            result = fit_sca(values, split, n_components=k, n_epochs=SCA_EPOCHS, seed=SEED)
            folder = OUTPUT / 'sca' / representation / f'k{k:02d}'
            folder.mkdir(parents=True, exist_ok=True)
            result['metrics'].to_csv(folder / 'metrics.csv', index=False)
            params = result['model'].params
            trial_activity = np.mean(np.abs(result['latent']), axis=1)
            np.savez_compressed(folder / 'fit.npz', trial_activity=trial_activity,
                                **{f'param_{name}': value for name, value in params.items()})
            for row in result['metrics'].to_dict('records'):
                sca_rows.append({'representation': representation, 'k': k, **row})

sca_summary = pd.DataFrame(sca_rows)
if len(sca_summary):
    sca_summary.to_csv(OUTPUT / 'sca_summary.csv', index=False)
    display(sca_summary)


## Comparison

The first comparison is held-out reconstruction as a function of component count. Reuse, stability across repeated fits/bootstrap samples, and cross-method component matching are subsequent Phase-2 analyses; they should be added only after the basic fits are numerically validated.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for method, table in [('FADA', fada_summary), ('SCA', sca_summary)]:
    if not len(table):
        continue
    test = table[table.split == 'test']
    for representation in ('q', 'f'):
        rows = test[test.representation == representation].sort_values('k')
        ax.plot(rows.k, rows.nmse, marker='o', label=f'{method} {representation}')
ax.set(xlabel='Number of components', ylabel='Held-out NMSE', title='Phase-2 held-out reconstruction')
ax.legend(frameon=False)
fig.savefig(OUTPUT / 'heldout_nmse.png')
plt.show()
